## Adapting Metrics From Statistical Games
- Games like basketball or baseball have massive amounts of data, dating back many years: [stats in other games verses soccer](https://phillysoccerpage.net/2017/04/06/on-advanced-stats-in-soccer-part-one-navigating-the-soccer-data-desert/#:~:text=The%20Current%20State%20of%20Advanced,detailed%20searchable%20and%20sortable%20data.)
- Because soccer is continuous and low-scoring, standard scoring stats sometimes fail to capture a player's true impact. By engineering cross-sport metrics we can perform advanced player evaluations

#### Basketball's "Usage Rate" (USG%)
In basketball, Usage Rate estimates the percentage of team plays "used" by a player while they are on the floor (ending in a shot, free throw, or turnover). In soccer, a player "uses" a possession by actively advancing the ball, shooting, or turning it over.

1. General Possession Usage RateThis measures a player's gravitational pull on the team's overall buildup play. Central midfielders  will dominate this metric.

$$\text{Possession USG\%} = \left( \frac{\text{Player Touches}}{\text{Team Total Touches (while player is on pitch)}} \right) \times 100$$

2. Attacking Usage Rate (The "Ball Dominance" Metric)
It measures how much of the team's final-third attacking bandwidth a specific winger or striker consumes.

$$\text{Attacking USG\%} = \frac{\text{Shots} + \text{Progressive Carries} + \text{Shot Creating Actions} + \text{Dispossessed}}{\text{Team Totals for same metrics}} \times 100$$

Why this matters: If a player has a large amount of Attacking USG% but a low xG/Assists output, they are a "high-volume, low-efficiency" player (the soccer equivalent of a chucker - someone who takes a lot of shots that are inefficient or poor quality shots, thereby wasting possession)

---

#### Baseball's WAR (Goals Added Above Replacement)
- Baseball's Wins Above Replacement (WAR) attempts to summarize a player's total contributions into one number. In soccer, goals are the ultimate currency, so we look at Net Goals Added.
- To calculate this, evaluate how much value a player adds compared to an average player at their position, heavily utilizing Expected Goals (xG) and Expected Assisted Goals (xAG) to remove the "luck" factor of teammates finishing poorly or goalkeepers making miraculous saves

$$\text{Offensive Goals Added} = (\text{npxG} + \text{xAG}) - \text{Positional Average}$$

---

#### Hockey's Corsi (Net xG Plus-Minus)
Hockey uses Corsi (shot attempts for vs. shot attempts against while a player is on the ice) to determine if a team controls the game when a specific player is deployed.
- In soccer, raw shot volume is less informative than shot quality. A Net xG Plus-Minus (often called On/Off xG)

$$\text{Net xG } \pm = (\text{Team xG per 90} - \text{Opponent xG per 90})_{\text{Player On}} - (\text{Team xG per 90} - \text{Opponent xG per 90})_{\text{Player Off}}$$

- This matters because isolates the player's true two-way impact
- If a center-back's defensive stats look average in your top_5_leagues_defense.csv
- But the team's Net xG Plus-Minus drops by a significant amount when they are subbed off or not playing, then this stat highlights that their positioning prevents high-quality chances, even if they aren't logging high tackle numbers (Virgil Van Dyke would be an example of this, someone frequently criticized for not defending but without him Liverpool concede much more, (spoken from a liverpool fan))

---

#### Basketball's True Shooting (True Finishing Efficiency)
- Basketball uses True Shooting Percentage to account for the different values of 3-pointers and free throws. 
- In soccer, a tap-in from 2 yards out and a screamer from 30 yards out both count as 1 goal, but they require entirely different finishing profiles.
- We can create a True Finishing Efficiency (TFE) metric that compares actual goal output to the mathematical probability of those shots going in
$$\text{TFE} = \frac{\text{Non-Penalty Goals (npG)}}{\text{Non-Penalty Expected Goals (npxG)}}$$

- TFE > 1.2: Elite finishing 
- TFE = 1.0: The player finishes exactly as an average professional would 
- TFE < 0.8: Poor finishing or bad luck

In [14]:
import pandas as pd
import numpy as np

General Formulas

In [15]:
def calculate_possession_usage(df):
    """
    Calculates Possession Usage %: The percentage of a team's touches 
    a player accounts for while they are on the pitch.
    Requires: 'top_5_leagues_possession.csv'
    """
    df = df.copy()
    
    # Calculate touches per 90 for the player
    df['Touches_per_90'] = df['Touches'] / df['90s']
    
    # Calculate the team's total touches and total 90s played
    # Note: 11 players are on the pitch, so Team 90s is roughly (Sum of all player 90s / 11)
    team_total_touches = df.groupby('Squad')['Touches'].transform('sum')
    team_total_90s = df.groupby('Squad')['90s'].transform('sum') / 11
    
    # Team touches per 90
    team_touches_per_90 = team_total_touches / team_total_90s
    
    # The Usage Rate (%)
    df['Possession_USG_pct'] = (df['Touches_per_90'] / team_touches_per_90) * 100
    
    return df

Usage Rate

In [16]:
def calculate_attacking_usage(df_standard, df_possession):
    """
    Calculates Attacking Usage %: The 'Ball Dominance' metric.
    Dynamically checks for available turnover columns to prevent KeyErrors.
    """
    df_standard = df_standard.copy()
    df_possession = df_possession.copy()
    
    anchor_cols = ['Player', 'Squad']
    # List all possible names for carries and turnovers
    possible_cols = ['Carries', 'CarMis', 'CarDis', 'Mis', 'Dis', 'PrgC'] 
    
    # Keep only the columns that actually exist in the possession dataframe
    available_cols = anchor_cols + [col for col in possible_cols if col in df_possession.columns]
    
    # Merge the standard data with our dynamically filtered possession data
    df = pd.merge(df_standard, df_possession[available_cols], on=['Player', 'Squad'], how='inner')
    
    # .get() will pull the column if it exists, otherwise it returns a Series of 0s
    shots = df.get('Sh', 0)
    
    # Standard usually has PrgC, but we check both dataframes just in case
    prog_carries = df.get('PrgC_x', df.get('PrgC_y', df.get('PrgC', 0))) 
    
    # Handle the different variations of turnover names
    miscontrols = df.get('CarMis', df.get('Mis', 0))
    dispossessed = df.get('CarDis', df.get('Dis', 0))
    
    df['Offensive_Actions'] = shots + prog_carries + miscontrols + dispossessed
    
    # Calculate the Usage Rate
    df['Off_Actions_per_90'] = df['Offensive_Actions'] / df['90s']
    
    team_total_actions = df.groupby('Squad')['Offensive_Actions'].transform('sum')
    team_total_90s = df.groupby('Squad')['90s'].transform('sum') / 11
    
    team_actions_per_90 = team_total_actions / team_total_90s
    
    # Avoid division by zero
    df['Attacking_USG_pct'] = np.where(team_actions_per_90 > 0, 
                                      (df['Off_Actions_per_90'] / team_actions_per_90) * 100, 
                                      0)
    return df

Offensive Goals Added (WAR)

In [17]:
def calculate_offensive_goals_added(df):
    """
    Calculates WAR equivalent: Offensive Goals Added above positional average.
    Requires: 'top_5_leagues_standard.csv'
    """
    df = df.copy()
    
    # Calculate the player's total expected offensive contribution per 90
    df['x_Offense_per_90'] = (df['npxG'] + df.get('xAG', df.get('xA', 0))) / df['90s']
    
    # Calculate the mean expected contribution for their specific position across the whole dataset
    # use transform('mean') to broadcast the positional average back to each player
    df['Positional_Avg_x_Offense'] = df.groupby('Pos')['x_Offense_per_90'].transform('mean')
    
    # Calculate Goals Added (per 90)
    df['Offensive_Goals_Added_per_90'] = df['x_Offense_per_90'] - df['Positional_Avg_x_Offense']
    
    # Calculate Total Goals Added for the season
    df['Total_Offensive_Goals_Added'] = df['Offensive_Goals_Added_per_90'] * df['90s']
    
    return df

True Finishing Efficiency (Basketball's True Shooting )

In [18]:
def calculate_true_finishing_efficiency(df):
    """
    Calculates True Finishing Efficiency (TFE).
    Requires: 'top_5_leagues_shooting.csv'
    """
    df = df.copy()
    
    # If npG isn't explicitly there, calculate it:
    if 'npG' not in df.columns and 'PK' in df.columns:
        df['npG'] = df['Gls'] - df['PK']
    elif 'npG' not in df.columns:
        df['npG'] = df['Gls'] # Fallback if PK data is missing 
        
    # Calculate TFE, handling division by zero for players with 0 npxG
    # We use np.where to assign np.nan to avoid skewing data with infinite values
    df['TFE'] = np.where(df['npxG'] > 0.1, df['npG'] / df['npxG'], np.nan)
    
    return df

Testing out the newly made statistics

In [19]:
import glob
import os

In [20]:
path = 'data/season_24-25'
all_files = glob.glob(os.path.join(path, "*.csv"))

# 2. Load into a dictionary: {'filename': dataframe}
soccer_data = {}
for filename in all_files:
    name = os.path.basename(filename).replace('.csv', '').replace('top_5_leagues_', '')
    soccer_data[name] = pd.read_csv(filename)
    print(f"Loaded: {name}")

Loaded: possession
Loaded: standard
Loaded: passing
Loaded: shooting
Loaded: defense


In [22]:
# 1. Extract the DataFrames
df_possession = soccer_data['possession']
df_standard = soccer_data['standard']
df_shooting = soccer_data['shooting']

# 2. Run the custom metric functions
df_possession = calculate_possession_usage(df_possession)
df_usage = calculate_attacking_usage(df_standard, df_possession)
df_efficiency = calculate_offensive_goals_added(df_standard)
df_finishing = calculate_true_finishing_efficiency(df_shooting)

comparison_df = df_usage[['Player', 'Squad', 'Pos', '90s', 'Attacking_USG_pct']].copy()

# Merge in Possession Usage directly from df_possession
comparison_df = comparison_df.merge(
    df_possession[['Player', 'Squad', 'Possession_USG_pct']], 
    on=['Player', 'Squad'],
    how='left'
)

# Merge in Goals Added from df_efficiency
comparison_df = comparison_df.merge(
    df_efficiency[['Player', 'Squad', 'Total_Offensive_Goals_Added']], 
    on=['Player', 'Squad'],
    how='left'
)

# Merge in True Finishing Efficiency from df_finishing
comparison_df = comparison_df.merge(
    df_finishing[['Player', 'Squad', 'TFE']], 
    on=['Player', 'Squad'],
    how='left'
)

# 4. Let's verify it worked by looking at Martin Ødegaard 
odegaard_stats = comparison_df[comparison_df['Player'] == 'Martin Ødegaard']
print(odegaard_stats)

# Let's also see the top 5 players in the league by Attacking Usage Rate (minimum 10 full 90s played)
top_usage = comparison_df[comparison_df['90s'] >= 10].sort_values(by='Attacking_USG_pct', ascending=False)
print("\nTop 5 Players by Attacking Usage Rate:")
print(top_usage.head())

               Player    Squad Pos   90s  Attacking_USG_pct  \
2802  Martin Ødegaard  Arsenal  MF  25.8           15.80909   

      Possession_USG_pct  Total_Offensive_Goals_Added   TFE  
2802           10.310619                     4.459503  0.75  

Top 5 Players by Attacking Usage Rate:
               Player            Squad    Pos   90s  Attacking_USG_pct  \
804     Chidera Ejuke          Sevilla     FW  10.7          44.604928   
594     Coba da Costa           Getafe  MF,FW  11.2          42.248377   
741       Jeremy Doku  Manchester City  FW,MF  16.8          42.000408   
2344      Léo Scienza       Heidenheim  FW,MF  12.8          35.995860   
1159  Gerrit Holtmann           Bochum  DF,FW  12.6          35.607397   

      Possession_USG_pct  Total_Offensive_Goals_Added       TFE  
804             9.623943                    -1.809987  1.818182  
594             9.675970                    -1.222750  0.909091  
741             8.258908                    -0.677207  2.307692  


In [24]:
output_folder = 'data/adv_stats_inc'
os.makedirs(output_folder, exist_ok=True)

comparison_df.to_csv(f'{output_folder}/master_player_comparison.csv', index=False)
print(f"Saved: master_player_comparison.csv")

dataframes_to_save = {
    'possession_with_usage': df_possession,
    'usage_ball_dominance': df_usage,
    'shooting_efficiency': df_finishing
}

for filename, df in dataframes_to_save.items():
    output_path = os.path.join(output_folder, f"{filename}.csv")
    df.to_csv(output_path, index=False)
    print(f"Saved: {filename}.csv")

print("\nAll stats exported to 'data/processed_metrics/'")

Saved: master_player_comparison.csv
Saved: possession_with_usage.csv
Saved: usage_ball_dominance.csv
Saved: shooting_efficiency.csv

All stats exported to 'data/processed_metrics/'
